# 05 — Zarr + SQL Views

**Zarr** supports chunked multidimensional arrays; **SQL** supports expressive filtering, aggregation, and joins. Together they are useful for climate-health cubes, gridded population, exposure rasters, ensembles, and simulation outputs.

This lab creates a tiny in-memory Zarr cube, then exposes a 2D slice as an SQL table.

In [ ]:
import micropip
await micropip.install(["zarr","xarray"])
import numpy as np, pandas as pd, xarray as xr, zarr, sqlite3
print("xarray",xr.__version__,"zarr",zarr.__version__)

In [ ]:
times=pd.date_range("2025-01-01",periods=12,freq="MS")
lat=np.linspace(-2,2,10); lon=np.linspace(28,32,12)
rng=np.random.default_rng(2)
arr=25+rng.normal(size=(12,10,12)).cumsum(axis=0)*.2
ds=xr.Dataset({"heat_index":(("time","lat","lon"),arr)},coords={"time":times,"lat":lat,"lon":lon})
store=zarr.storage.MemoryStore(); ds.to_zarr(store,mode="w")
cube=xr.open_zarr(store); cube

In [ ]:
slice_df=cube.heat_index.isel(time=6).to_dataframe().reset_index()
con=sqlite3.connect(":memory:"); slice_df.to_sql("climate",con,index=False)
sql="SELECT ROUND(lat,1) AS lat_band, COUNT(*) AS n, ROUND(AVG(heat_index),2) AS avg_heat FROM climate WHERE heat_index > 24.8 GROUP BY ROUND(lat,1) ORDER BY avg_heat DESC"
pd.read_sql_query(sql,con).head(10)

For larger browser deployments, consider Zarr-compatible object storage plus DuckDB-Wasm or a governed server-side query service. Data classification comes before convenience.